# V2a-RSN 220127_F4_run2 — Linking Notebook: Partial Correlation Baseline

Prerequisites: `graph_states`, `community`, `manifold`, and `correlation_partial` `connectivity` stages for the selected run, plus the dataset `shared/recording` checkpoint.
This notebook preserves the anchor-aware embargo and activity-covariate controls in the resumed stage artifact.
This lane is an undirected baseline. Use it for comparison, not for directional driver claims.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.

Environment: select the project Python 3.14 `.venv` kernel. Install the locked packages with `uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks` from the repository root. The setup cell checks installed package versions; no upstream repository clone is required. Stored outputs may predate the backend migration; rerun the notebook to produce results with the current backend.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


if sys.version_info[:2] != (3, 14):
    raise RuntimeError(
        "Select the project Python 3.14 .venv kernel before running this notebook. "
        f"Current interpreter: {sys.executable}"
    )
_BACKEND_DISTRIBUTIONS = ("fastcgc", "tigramite", "scikit-learn", "kmedoids", "igraph", "leidenalg")
try:
    _BACKEND_VERSIONS = {name: version(name) for name in _BACKEND_DISTRIBUTIONS}
except PackageNotFoundError as error:
    raise RuntimeError(
        "The notebook kernel is missing a required installed backend. Run "
        "uv sync --python 3.14 --locked --extra dev --extra csl --extra community --extra notebooks "
        "from the repository root, then restart the project .venv kernel."
    ) from error
print("Notebook interpreter:", sys.executable)
print("Installed backends:", _BACKEND_VERSIONS)


In [ ]:
from notebooks._shared import PROJECT_ROOT, make_run, print_notebook_header, print_stage_status, run_linking

DATASET_ID = "220127_F4_run2"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220127_F4_run2"
RUN_ID = "reference"
METHOD = "correlation_partial"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
FORCE = False

LINKING_CFG = {
    "behavior_keys": ["continuous"],
    "n_null": 1000,
    "n_bins": 5,
    "max_lag": 10,
    "positive_lag": 1,
    "n_folds": 5,
    "embargo": 4,
    "group_by": "recording",
    "lag_extension": 0,
    "preprocessing_past_support": 0,
    "preprocessing_future_support": 0,
    "allow_global_preprocessing": False,
    "null_kind": "circular_shift",
    "block_length": 8,
    "seed": 42,
}

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} linking",
    expected_stages=["recording_input", "linking"],
    force=FORCE,
)


In [ ]:
report = run_linking(
    run,
    recording_path=RECORDING_PATH,
    linking_cfg=LINKING_CFG,
    force=FORCE,
)
if report.get("status") == "unavailable":
    print("Linking unavailable:", report["reason"])
else:
    print(len(report["decoding"]))
    print(report["splitter"])
    print(report["positive_lag_incremental"].keys())
    print(report["effectome_to_future_manifold"])
print_stage_status(run)
